# Customer Churn Prediction — Retail Athletic Demo

Predict which customers are likely to churn from purchase behaviour and engagement, then operationalise the model in Snowflake:

1. Build a feature view in SQL (`CHURN_FEATURES_V`)
2. Train an XGBoost classifier and compare it to the legacy rule-based score
3. Log the model to the **Snowflake Model Registry**
4. Run **batch inference in-warehouse** with `mv.run()` → `CHURN_PREDICTIONS`
5. Attach a **Model Monitor** (performance + drift vs. a training baseline)

**Prereqs:** run `sql/01`–`02` first. Python env: `pip install "snowflake-ml-python>=1.7.1" xgboost scikit-learn pandas`.
Also runs as-is in a Snowflake Notebook (it picks up the active session).

In [ ]:
# ---- Configuration: edit these ----
CONNECTION_NAME = "default"          # connection name in ~/.snowflake/connections.toml (ignored inside Snowflake Notebooks)
DATABASE  = "RETAIL_ATHLETIC_DEMO"
SCHEMA    = "PUBLIC"
WAREHOUSE = "COMPUTE_WH"
MODEL_NAME, VERSION_NAME, MONITOR_NAME = "CHURN_PREDICTION_MODEL", "V1", "CHURN_MODEL_MONITOR"

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()                      # inside a Snowflake Notebook
except Exception:
    session = Session.builder.config("connection_name", CONNECTION_NAME).create()

session.use_warehouse(WAREHOUSE)
session.use_schema(f"{DATABASE}.{SCHEMA}")
print(session.get_current_account(), session.get_current_role(), session.get_fully_qualified_current_schema())

## 1. Feature view
All features are computed in SQL so training and in-warehouse scoring use exactly the same definitions.
The label `CHURNED` is `CHURN_RISK_SCORE >= 0.5`. `LEGACY_CHURN_SCORE` is the existing rule-based score, kept only for comparison (not a feature).

In [ ]:
session.sql(f"""
CREATE OR REPLACE VIEW {DATABASE}.{SCHEMA}.CHURN_FEATURES_V AS
SELECT
    c.CUSTOMER_ID,
    m.DAYS_SINCE_LAST_PURCHASE::FLOAT                                   AS DAYS_SINCE_LAST_PURCHASE,
    m.PURCHASE_FREQUENCY::FLOAT                                         AS PURCHASE_FREQUENCY,
    m.AVG_ORDER_VALUE::FLOAT                                            AS AVG_ORDER_VALUE,
    m.TOTAL_SPEND_12M::FLOAT                                            AS TOTAL_SPEND_12M,
    m.PRODUCT_DIVERSITY_SCORE::FLOAT                                    AS PRODUCT_DIVERSITY_SCORE,
    m.CHANNEL_DIVERSITY_SCORE::FLOAT                                    AS CHANNEL_DIVERSITY_SCORE,
    m.RETURN_RATE::FLOAT                                                AS RETURN_RATE,
    m.EMAIL_ENGAGEMENT_RATE::FLOAT                                      AS EMAIL_ENGAGEMENT_RATE,
    m.WEB_VISITS_30D::FLOAT                                             AS WEB_VISITS_30D,
    c.NPS_SCORE::FLOAT                                                  AS NPS_SCORE,
    c.TOTAL_ORDERS::FLOAT                                               AS TOTAL_ORDERS,
    c.LIFETIME_VALUE::FLOAT                                             AS LIFETIME_VALUE,
    DATEDIFF('day', c.FIRST_PURCHASE_DATE, m.FEATURE_DATE)::FLOAT       AS TENURE_DAYS,
    IFF(c.OPT_IN_EMAIL, 1, 0)::FLOAT                                    AS OPT_IN_EMAIL,
    IFF(c.OPT_IN_SMS, 1, 0)::FLOAT                                      AS OPT_IN_SMS,
    IFF(c.PREFERRED_CHANNEL = 'Retail Store', 1, 0)::FLOAT              AS PREFERS_RETAIL,
    IFF(c.REGION = 'International', 1, 0)::FLOAT                        AS IS_INTERNATIONAL,
    IFF(c.CHURN_RISK_SCORE >= 0.5, 1, 0)                                AS CHURNED,
    m.CHURN_PROBABILITY::FLOAT                                          AS LEGACY_CHURN_SCORE
FROM {DATABASE}.{SCHEMA}.DIM_CUSTOMER c
JOIN {DATABASE}.{SCHEMA}.ML_CUSTOMER_FEATURES m ON m.CUSTOMER_ID = c.CUSTOMER_ID
""").collect()

df = session.table("CHURN_FEATURES_V").to_pandas()
FEATURES = [c for c in df.columns if c not in ("CUSTOMER_ID", "CHURNED", "LEGACY_CHURN_SCORE")]
print(df.shape, "| churn rate:", round(df.CHURNED.mean(), 3))
df.head()

## 2. Train XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df.CHURNED)
X_train, y_train = train_df[FEATURES], train_df.CHURNED
X_test,  y_test  = test_df[FEATURES],  test_df.CHURNED

model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
auc, ap = roc_auc_score(y_test, proba), average_precision_score(y_test, proba)
legacy_auc = roc_auc_score(y_test, test_df.LEGACY_CHURN_SCORE)
print(f"XGBoost ROC AUC: {auc:.3f}   (legacy rule-based score: {legacy_auc:.3f})   PR AUC: {ap:.3f}")
print(classification_report(y_test, (proba >= 0.5).astype(int), target_names=["Retained", "Churned"]))

pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10)

## 3. Log to the Model Registry
Re-running this notebook replaces the model (and its monitor), so the demo is repeatable.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

session.sql(f"DROP MODEL MONITOR IF EXISTS {MONITOR_NAME}").collect()
if MODEL_NAME in reg.show_models()["name"].str.upper().values:
    reg.delete_model(MODEL_NAME)

mv = reg.log_model(
    model,
    model_name=MODEL_NAME,
    version_name=VERSION_NAME,
    sample_input_data=X_train.head(100),
    target_platforms=["WAREHOUSE"],
    task=type_hints.Task.TABULAR_BINARY_CLASSIFICATION,
    comment="XGBoost churn classifier on behavioural + engagement features",
    metrics={"roc_auc": round(auc, 4), "pr_auc": round(ap, 4), "legacy_roc_auc": round(legacy_auc, 4)},
)
reg.get_model(MODEL_NAME).show_versions()

## 4. Batch inference in Snowflake
`mv.run()` scores the Snowpark DataFrame inside the warehouse — no data leaves Snowflake.

In [ ]:
features_sp = session.table("CHURN_FEATURES_V")
scored = mv.run(features_sp, function_name="predict_proba")

# predict_proba returns one column per class; take the probability of class 1
prob_col = [c for c in scored.columns if c.strip('"').upper().endswith("OUTPUT_FEATURE_1")][0]

predictions = (scored
    .with_column("CHURN_SCORE", F.round(F.col(prob_col), 4))
    .with_column("CHURN_PREDICTION", F.iff(F.col("CHURN_SCORE") >= 0.5, 1, 0))
    .with_column("SCORED_AT", F.current_timestamp().cast("TIMESTAMP_NTZ")))

predictions.select("CUSTOMER_ID", "CHURN_SCORE", "CHURN_PREDICTION", "SCORED_AT") \
           .write.mode("overwrite").save_as_table("CHURN_PREDICTIONS")

session.sql("""
SELECT CHURN_PREDICTION, COUNT(*) AS customers, ROUND(AVG(CHURN_SCORE), 3) AS avg_score
FROM CHURN_PREDICTIONS GROUP BY 1 ORDER BY 1
""").to_pandas()

**Demo moment:** high-value customers the model flags as likely to churn — a ready-made retention list.

In [ ]:
session.sql("""
SELECT c.FIRST_NAME || ' ' || c.LAST_NAME AS customer, c.SEGMENT, c.PREFERRED_CHANNEL,
       c.LIFETIME_VALUE, f.CLV_PREDICTED_12M, p.CHURN_SCORE
FROM CHURN_PREDICTIONS p
JOIN DIM_CUSTOMER c         ON c.CUSTOMER_ID = p.CUSTOMER_ID
JOIN ML_CUSTOMER_FEATURES f ON f.CUSTOMER_ID = p.CUSTOMER_ID
WHERE p.CHURN_PREDICTION = 1
ORDER BY c.LIFETIME_VALUE DESC
LIMIT 15
""").to_pandas()

## 5. Model Monitor
The monitor needs a **source** table (features + prediction + actual + timestamp) and, for drift, a **baseline**.
- Baseline = the training rows, scored by the model.
- Source = the held-out rows, with scoring timestamps spread over the last 4 weeks so the monitor has a daily time series.

In [ ]:
model_cols = FEATURES + ["CHURN_SCORE", "CHURN_PREDICTION", "CHURNED"]

def monitor_frame(ids):
    return (predictions
            .filter(F.col("CUSTOMER_ID").isin(list(ids)))
            .select("CUSTOMER_ID", *model_cols,
                    F.dateadd("day", -(F.abs(F.hash(F.col("CUSTOMER_ID"))) % 28),
                              F.to_timestamp_ntz(F.lit("2025-09-30"))).alias("SCORED_AT")))

monitor_frame(train_df.CUSTOMER_ID).write.mode("overwrite").save_as_table("CHURN_MONITOR_BASELINE")
monitor_frame(test_df.CUSTOMER_ID).write.mode("overwrite").save_as_table("CHURN_MONITOR_SOURCE")

session.sql(f"""
CREATE OR REPLACE MODEL MONITOR {MONITOR_NAME} WITH
    MODEL = {MODEL_NAME}
    VERSION = '{VERSION_NAME}'
    FUNCTION = 'predict'
    SOURCE = {DATABASE}.{SCHEMA}.CHURN_MONITOR_SOURCE
    BASELINE = {DATABASE}.{SCHEMA}.CHURN_MONITOR_BASELINE
    WAREHOUSE = {WAREHOUSE}
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = SCORED_AT
    ID_COLUMNS = ('CUSTOMER_ID')
    PREDICTION_SCORE_COLUMNS = ('CHURN_SCORE')
    PREDICTION_CLASS_COLUMNS = ('CHURN_PREDICTION')
    ACTUAL_CLASS_COLUMNS = ('CHURNED')
""").collect()

session.sql(f"SHOW MODEL MONITORS LIKE '{MONITOR_NAME}'").to_pandas()[["name", "monitor_state"]]

In [ ]:
# Monitor metrics are computed on refresh. If this returns no rows, wait a minute and re-run.
session.sql(f"""
SELECT * FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    '{MONITOR_NAME}', 'ROC_AUC', 'DAY', '2025-09-01'::TIMESTAMP_NTZ, '2025-10-01'::TIMESTAMP_NTZ))
ORDER BY 1
""").to_pandas()

## Summary
| Object | Name |
|---|---|
| Feature view | `CHURN_FEATURES_V` |
| Registered model | `CHURN_PREDICTION_MODEL` / `V1` |
| Batch predictions | `CHURN_PREDICTIONS` |
| Monitor | `CHURN_MODEL_MONITOR` (source `CHURN_MONITOR_SOURCE`, baseline `CHURN_MONITOR_BASELINE`) |

In Snowsight: **AI & ML → Models → CHURN_PREDICTION_MODEL → Monitors** for the drift and performance dashboards.